In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.Requirement import Requirement
from EmpenageSizing.EmpenageFinder import EmpenageFinder

# Loading the pre-computed planforms and the fuselage

In [2]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

In [3]:
assumptions = Assumptions()

#TODO load the fuselage here
main_gear_bay = Bay(surface_wetted=np.pi*0.02*0.5, length=0.5, diameter=0.02)
engine_bay = Bay(surface_wetted=np.pi*0.02*0.5, length=0.5, diameter=0.02)
main_gear = LandingGear(wheel_width=assumptions.main_gear_width_wheel, exposed_height=0.02, wheel_diameter=0.03, strut_width=0.01)
nose_gear = LandingGear(wheel_width=assumptions.main_gear_width_wheel, exposed_height=0.02, wheel_diameter=0.03, strut_width=0.01)
fuselage = Fuselage(surface_wetted=np.pi*0.03*3.4, length_total=3.4, diameter_max=0.03, upsweep=np.pi/12, base_area=np.pi/4*0.007**2)
fixed = Fixed(mass=25., fuel_mass=10., x_cg_min=1.6, x_cg_max=2., x_tail_cone=2.3, z_cg=0.04, z_tail_cone=0.02, z_wing=0.05, x_LE_canard=0.01,
              x_LE_wing=1.6, x_LE_tail=3.25, x_nose_gear=0.1, x_main_gear=1.8, y_main_gear=0.02, fuselage=fuselage, nose_gear=nose_gear,
              main_gear=main_gear, gear_bay=main_gear_bay, engine_bay=engine_bay)

In [4]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

c:\marek\studia\Y3\DSE\repo\DSE-28\src_final\Drag\Component.py:19: RuntimeWarning: invalid value encountered in scalar divide
  laminar_comp = self.laminar_fraction*1.328/np.sqrt(reynolds_)
c:\marek\studia\Y3\DSE\repo\DSE-28\src_final\Drag\Component.py:20: RuntimeWarning: divide by zero encountered in log10
  turbulent_comp = (1-self.laminar_fraction)*.455/(np.log10(reynolds_)**2.58*(1+.144*mach**2)**.65)


# Creating full Aircraft objects

In [5]:
aircraft:list[Aircraft] = list()

for planform_recovered in plaforms_recovered:
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]
    planform_stable = planform_recovered[2]

    aircraft_planforms = [main_wing, ] #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

# Checking if requirements are met

In [6]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq()
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear"
]

In [7]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

c:\marek\studia\Y3\DSE\repo\DSE-28\src_final\MatchingDiagram\MatchingDiagramJet.py:45: RuntimeWarning: invalid value encountered in sqrt
  CL_optimal = np.sqrt(CD0*inviscid_ratio)
c:\marek\studia\Y3\DSE\repo\DSE-28\src_final\MatchingDiagram\MatchingDiagramJet.py:47: RuntimeWarning: invalid value encountered in sqrt
  numerator = (2 * np.sqrt(CD0 / inviscid_ratio) + sin_gradient) * engine_coefficient * beta
c:\marek\studia\Y3\DSE\repo\DSE-28\src_final\MatchingDiagram\MatchingDiagramJet.py:61: RuntimeWarning: invalid value encountered in sqrt
  proportionality = 1.15 * np.sqrt(self.engine_inoperative_coefficient / field_length / .85 / CONSTANTS.G0 / denisty / inviscid_ratio)


TypeError: FuelReq.assess() missing 2 required positional arguments: 'constants' and 'assumptions'